# LeetCode 1963 — Minimum Number of Swaps to Make the String Balanced bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-swaps-to-make-the-string-balanced/ · Độ khó: **Medium** · Chủ đề: Two Pointers, String, Stack, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minSwaps(self, s: str) -> int:
        ...
```

Notebook đi từ **BFS trên mọi xâu đạt được bằng hoán đổi** (vét cạn) → **mô phỏng tham lam hai con trỏ** (gặp `]` thừa thì đổi với `[` xa nhất bên phải) → **ngăn xếp loại bỏ cặp khớp** → **số dư tiền tố nhỏ nhất** với công thức `⌈m/2⌉`.

## 1. Mục tiêu học

1. Ôn lại định nghĩa **xâu ngoặc cân bằng** và tiêu chí tương đương: mọi **số dư tiền tố** (`[` là +1, `]` là −1) đều `≥ 0` và tổng bằng 0.
2. Hiểu sau khi loại các cặp `[]` khớp, phần còn lại luôn có dạng `]]…][[…[` với `m` dấu mỗi loại.
3. Chứng minh đáp án là `⌈m/2⌉`: mỗi lần hoán đổi sửa được **tối đa 2** cặp lỗi.
4. Cài đặt bằng tham lam hai con trỏ, ngăn xếp, và một biến đếm; tránh lỗi làm tròn.
5. Kiểm thử bằng BFS vét cạn với mọi xâu cân số ngoặc độ dài nhỏ.

## 2. Đề bài & ví dụ

Cho xâu `s` độ dài chẵn `n`, gồm đúng `n/2` dấu `'['` và `n/2` dấu `']'`.

Một xâu **cân bằng** nếu: là xâu rỗng; hoặc là ghép `AB` của hai xâu cân bằng; hoặc có dạng `[C]` với `C` cân bằng.

Bạn được **hoán đổi** hai ký tự ở hai vị trí bất kỳ, bao nhiêu lần tùy ý. Trả về **số lần hoán đổi ít nhất** để `s` cân bằng.

**Ví dụ 1:** `s = "][]["` → `1` (đổi vị trí 0 và 3 được `"[[]]"`).

**Ví dụ 2:** `s = "]]][[["` → `2` (đổi 0↔4 được `"[]][]["`, rồi 1↔5 được `"[[][]]"`).

**Ví dụ 3:** `s = "[]"` → `0`.

**Ràng buộc:** `2 ≤ n ≤ 10⁶`, `n` chẵn; số `'['` bằng số `']'` bằng `n/2`.

In [1]:
import random
import time
from collections import deque
from itertools import combinations
from typing import List

examples = [("][][", 1), ("]]][[[", 2), ("[]", 0)]

def is_balanced(s):
    bal = 0
    for c in s:
        bal += 1 if c == '[' else -1
        if bal < 0:
            return False
    return bal == 0

def swap(s, i, j):
    t = list(s); t[i], t[j] = t[j], t[i]
    return ''.join(t)

print(is_balanced("[[]]"), is_balanced("][]["))
print("ví dụ 2:", swap("]]][[[", 0, 4), "->", swap(swap("]]][[[", 0, 4), 1, 5), is_balanced(swap(swap("]]][[[", 0, 4), 1, 5)))

True False
ví dụ 2: []][][ -> [[][]] True


## 3. Kiến thức nền

### 3.1 Số dư tiền tố (prefix balance)

Duyệt xâu, `[` cộng 1, `]` trừ 1. Xâu cân bằng ⇔ số dư **không bao giờ âm** và kết thúc bằng 0. Giá trị **âm nhỏ nhất** cho biết có bao nhiêu `]` "thừa" chưa được ghép.

In [2]:
def prefix_balances(s):
    out, bal = [], 0
    for c in s:
        bal += 1 if c == '[' else -1
        out.append(bal)
    return out

for s in ["[[]]", "][][", "]]][[["]:
    pb = prefix_balances(s)
    print(f"{s}: {pb}, min = {min(pb)}")

[[]]: [1, 2, 1, 0], min = 0
][][: [-1, 0, -1, 0], min = -1
]]][[[: [-1, -2, -3, -2, -1, 0], min = -3


### 3.2 Ngăn xếp loại bỏ cặp khớp

Gặp `[` thì đẩy vào; gặp `]` mà đỉnh là `[` thì bỏ cặp đó ra. Phần còn lại là các ngoặc **không khớp**.

In [3]:
def unmatched(s):
    st = []
    for c in s:
        if c == ']' and st and st[-1] == '[':
            st.pop()
        else:
            st.append(c)
    return ''.join(st)

for s in ["][][", "]]][[[", "[]][][", "]][[]["]:
    print(f"{s} -> còn lại {unmatched(s)!r}")

][][ -> còn lại ']['
]]][[[ -> còn lại ']]][[['
[]][][ -> còn lại ']['
]][[][ -> còn lại ']][['


### 3.3 Làm tròn lên với số nguyên

`⌈m/2⌉ = (m + 1) // 2` (với `m ≥ 0`). Python còn có `-(-m // 2)` hay `(m + 1) >> 1`.

In [4]:
print([(m, (m + 1) // 2, -(-m // 2), (m + 1) >> 1) for m in range(6)])

[(0, 0, 0, 0), (1, 1, 1, 1), (2, 1, 1, 1), (3, 2, 2, 2), (4, 2, 2, 2), (5, 3, 3, 3)]


### 3.4 BFS cho "số thao tác ít nhất"

Mỗi xâu là một đỉnh, mỗi hoán đổi là một cạnh; BFS từ `s` cho số hoán đổi ít nhất tới xâu cân bằng đầu tiên gặp. Chỉ khả thi với `n` nhỏ.

In [5]:
print("số xâu độ dài 8 có 4 '[' :", sum(1 for _ in combinations(range(8), 4)))

số xâu độ dài 8 có 4 '[' : 70


## 4. Ý tưởng nền tảng / trực giác

Bỏ hết các cặp `[]` đã khớp (chúng không cần đụng tới). Phần còn lại luôn có dạng `]^m [^m` — vì nếu còn một `[` đứng trước một `]` thì chúng đã khớp với nhau.

Với `]^m [^m`:
- `m = 1`: `][` → đổi một lần thành `[]`.
- `m = 2`: `]][[` → đổi `]` đầu (vị trí 0) với `[` cuối (vị trí 3) được `[][]` — **một** lần đổi sửa cả hai cặp lỗi.
- Mỗi lần hoán đổi một `]` ở nửa trái với một `[` ở nửa phải sửa được **hai** cặp lỗi (ngoại trừ khi chỉ còn một cặp). ⇒ cần `⌈m/2⌉` lần.

Tính tay ví dụ 2 `]]][[[` (`m = 3`): đổi `0 ↔ 5` ⇒ `[]][[]`, còn lại một cặp lỗi `][` ở giữa ⇒ đổi thêm một lần ⇒ tổng `2 = ⌈3/2⌉`.

In [6]:
for t in ["][", "]][[", "]]][[[", "]]]][[[["]:
    m = len(t) // 2
    fixed = swap(t, 0, len(t) - 1)
    print(f"{t}: m={m}, đổi đầu<->cuối -> {fixed}, phần không khớp còn {unmatched(fixed)!r}, công thức ceil(m/2) = {(m + 1) // 2}")

][: m=1, đổi đầu<->cuối -> [], phần không khớp còn '', công thức ceil(m/2) = 1
]][[: m=2, đổi đầu<->cuối -> [][], phần không khớp còn '', công thức ceil(m/2) = 1
]]][[[: m=3, đổi đầu<->cuối -> []][[], phần không khớp còn '][', công thức ceil(m/2) = 2
]]]][[[[: m=4, đổi đầu<->cuối -> []]][[[], phần không khớp còn ']][[', công thức ceil(m/2) = 2


Vẽ "núi" số dư tiền tố cho `]]][[[` trước và sau mỗi lần hoán đổi tối ưu: đáy núi (giá trị nhỏ nhất) đi từ `−3` lên `−1` rồi lên `0` — mỗi lần đổi nâng đáy **2 đơn vị** (trừ lần cuối).

In [7]:
def mountain(s):
    pb = [0] + prefix_balances(s)
    lo, hi = min(pb), max(pb)
    for level in range(hi, lo - 1, -1):
        print(f"  {level:>2} | " + ''.join('●' if v == level else ('─' if level == 0 else ' ') for v in pb))
    print(f"     min = {lo}")

for t in ["]]][[[", "[]][[]", "[][[]]"]:
    print(t); mountain(t)

]]][[[
   0 | ●─────●
  -1 |  ●   ● 
  -2 |   ● ●  
  -3 |    ●   
     min = -3
[]][[]
   1 |  ●   ● 
   0 | ●─●─●─●
  -1 |    ●   
     min = -1
[][[]]
   2 |     ●  
   1 |  ● ● ● 
   0 | ●─●───●
     min = 0


Bảng trên cho thấy: mỗi lần đổi `]` **ngoài cùng bên trái** với `[` **ngoài cùng bên phải**, số cặp lỗi giảm **2** (từ `m` còn `m − 2`), trừ trường hợp `m = 1` giảm 1. ⇒ `⌈m/2⌉`.

## 5. Cách 1 — Brute Force / baseline: BFS trên mọi xâu

In [8]:
def min_swaps_bfs(s):
    if is_balanced(s):
        return 0
    n = len(s)
    dist = {s: 0}
    q = deque([s])
    while q:
        cur = q.popleft()
        for i in range(n):
            for j in range(i + 1, n):
                if cur[i] != cur[j]:
                    nxt = swap(cur, i, j)
                    if nxt not in dist:
                        dist[nxt] = dist[cur] + 1
                        if is_balanced(nxt):
                            return dist[nxt]
                        q.append(nxt)
    return -1

for s_, exp in examples:
    print(s_, "->", min_swaps_bfs(s_), "(kỳ vọng", exp, ")")

][][ -> 1 (kỳ vọng 1 )
]]][[[ -> 2 (kỳ vọng 2 )
[] -> 0 (kỳ vọng 0 )


**Độ phức tạp:** số xâu cân số ngoặc là `C(n, n/2)` (~`10²⁹⁹⁹⁹⁷` khi `n = 10⁶`), mỗi xâu `O(n²)` láng giềng — chỉ chạy được với `n ≤ 12`.

**Nút thắt:** hầu hết hoán đổi là vô ích; ta cần biết **chính xác** cái gì "hỏng" trong xâu — đó là các `]` thừa ở tiền tố.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `m` = số `]` không khớp (= `−min(0, min tiền tố)`, hoặc độ dài nửa phần còn lại sau khi loại cặp khớp). Đáp án `= ⌈m/2⌉`.

**Quy tắc tham lam (để dựng lời giải):** duyệt trái sang phải giữ `bal`; **nếu** gặp `]` làm `bal` âm **thì** hoán đổi nó với `[` **xa nhất bên phải** chưa dùng ⇒ `bal` tăng thành `+1`, đếm một lần.

**Bất biến:** khi duyệt xong tiền tố `s[0..i]`, tiền tố đó (sau các hoán đổi đã làm) có số dư `≥ 0`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam hai con trỏ (mô phỏng hoán đổi thật)

In [9]:
def min_swaps_greedy(s):
    t = list(s)
    j = len(t) - 1                 # con trỏ tìm '[' xa nhất bên phải
    bal = swaps = 0
    for i in range(len(t)):
        bal += 1 if t[i] == '[' else -1
        if bal < 0:
            while t[j] != '[':
                j -= 1
            t[i], t[j] = t[j], t[i]   # đổi ']' tại i với '[' tại j
            swaps += 1
            bal = 1                   # i giờ là '[': -1 + 2
    return swaps

for s_, exp in examples:
    print(s_, "->", min_swaps_greedy(s_), "(kỳ vọng", exp, ")")

][][ -> 1 (kỳ vọng 1 )
]]][[[ -> 2 (kỳ vọng 2 )
[] -> 0 (kỳ vọng 0 )


Vì chỉ đổi khi `bal < 0` và đổi với `[` ở **phía sau** `i` (con trỏ `j` không bao giờ đi qua `i` — luôn còn `[` phía sau khi tiền tố đang thiếu `[`), mỗi vị trí được xét một lần ⇒ `O(n)` thời gian, `O(n)` bộ nhớ (list).

### 7.2 Cách 3 — Ngăn xếp loại cặp khớp

In [10]:
def min_swaps_stack(s):
    st = []
    for c in s:
        if c == ']' and st and st[-1] == '[':
            st.pop()
        else:
            st.append(c)
    m = len(st) // 2                 # phần còn lại ]^m [^m
    return (m + 1) // 2

for s_, exp in examples:
    print(s_, "->", min_swaps_stack(s_), "(kỳ vọng", exp, ")")

][][ -> 1 (kỳ vọng 1 )
]]][[[ -> 2 (kỳ vọng 2 )
[] -> 0 (kỳ vọng 0 )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### 7.3 Cách 4 — Một biến đếm `[` chưa khớp (lời giải chính)

Không cần lưu ngăn xếp: chỉ đếm số `[` đang mở `open_`. Gặp `]` mà `open_ > 0` thì ghép (giảm), ngược lại `]` đó là thừa — **không** cần đếm vì số `]` thừa bằng số `[` thừa còn lại cuối cùng. Cuối cùng `m = open_`.

In [11]:
def min_swaps_counter(s):
    open_ = 0
    for c in s:
        if c == '[':
            open_ += 1
        elif open_:
            open_ -= 1
    return (open_ + 1) // 2

def min_swaps_min_prefix(s):
    bal = low = 0
    for c in s:
        bal += 1 if c == '[' else -1
        low = min(low, bal)
    m = -low
    return (m + 1) // 2

for s_, exp in examples:
    print(s_, "->", min_swaps_counter(s_), min_swaps_min_prefix(s_), "(kỳ vọng", exp, ")")

][][ -> 1 1 (kỳ vọng 1 )
]]][[[ -> 2 2 (kỳ vọng 2 )
[] -> 0 0 (kỳ vọng 0 )


Hai cách viết tương đương: số `[` thừa cuối cùng = số `]` thừa = `−min(tiền tố)` (vì tổng bằng 0). **Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ.

## 8. TRACE từng bước

Trace cách 2 (tham lam) trên `s = "]]][[["`:

| i | t trước | bal sau ký tự | âm? | hoán đổi | t sau | swaps |
|---|---|---|---|---|---|---|
| 0 | `]]][[[` | −1 | ✔ | 0↔5 | `[]][[]` | 1 |
| 1 | | 0 | | | | 1 |
| 2 | | −1 | ✔ | 2↔4 | `[][[]]` | 2 |
| 3..5 | | 0..1 | | | | 2 |

In [12]:
def trace_greedy(s):
    t = list(s); j = len(t) - 1; bal = swaps = 0
    for i in range(len(t)):
        bal += 1 if t[i] == '[' else -1
        note = ""
        if bal < 0:
            while t[j] != '[':
                j -= 1
            t[i], t[j] = t[j], t[i]; swaps += 1; bal = 1
            note = f" -> âm! đổi {i}<->{j}"
        print(f"  i={i}: bal={bal:>2}{note:<18} t={''.join(t)} swaps={swaps}")
    print("  kết quả:", ''.join(t), "cân bằng:", is_balanced(''.join(t)))
    return swaps

trace_greedy("]]][[[")

  i=0: bal= 1 -> âm! đổi 0<->5  t=[]][[] swaps=1
  i=1: bal= 0                   t=[]][[] swaps=1
  i=2: bal= 1 -> âm! đổi 2<->4  t=[][[]] swaps=2
  i=3: bal= 2                   t=[][[]] swaps=2
  i=4: bal= 1                   t=[][[]] swaps=2
  i=5: bal= 0                   t=[][[]] swaps=2
  kết quả: [][[]] cân bằng: True


2

In [13]:
def trace_counter(s):
    open_ = 0
    for i, c in enumerate(s):
        if c == '[':
            open_ += 1; act = "mở"
        elif open_:
            open_ -= 1; act = "ghép"
        else:
            act = "']' thừa"
        print(f"  i={i} '{c}': {act:<9} open={open_}")
    print(f"  m = {open_} -> (m+1)//2 = {(open_ + 1) // 2}")
    return (open_ + 1) // 2

trace_counter("][][")

  i=0 ']': ']' thừa  open=0
  i=1 '[': mở        open=1
  i=2 ']': ghép      open=0
  i=3 '[': mở        open=1
  m = 1 -> (m+1)//2 = 1


1

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minSwaps(self, s: str) -> int:
        open_ = 0                     # số '[' chưa được ghép
        for c in s:
            if c == '[':
                open_ += 1
            elif open_:
                open_ -= 1
        return (open_ + 1) // 2       # ceil(m / 2)

for s_, exp in examples:
    assert Solution().minSwaps(s_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `m` là số `]` không khớp (= `−min` số dư tiền tố, với `min` tính cả tiền tố rỗng `0`).

**Chặn dưới.** Một hoán đổi chỉ thay đổi hai vị trí `i < j`. Hoán đổi `]` (ở `i`) với `[` (ở `j`) làm số dư của mọi tiền tố kết thúc trong `[i, j)` **tăng đúng 2**, các tiền tố khác không đổi; hoán đổi theo chiều ngược lại làm giảm; hoán đổi hai ký tự giống nhau vô tác dụng. Vậy mỗi hoán đổi tăng `min` tiền tố **tối đa 2** ⇒ cần ít nhất `⌈m/2⌉` lần để `min` từ `−m` lên `0`.

**Đạt được.** Xét phần không khớp `]^m [^m` (các cặp đã khớp xen giữa không ảnh hưởng). Đổi `]` **đầu tiên** với `[` **cuối cùng** cho `[ ]^{m−1} [^{m−1} ]`: dấu `[` mới ở đầu khớp với `]` đầu tiên của phần giữa, dấu `]` mới ở cuối khớp với `[` cuối cùng của phần giữa ⇒ phần không khớp còn `]^{m−2} [^{m−2}`. Mỗi lần đổi giảm `m` đi 2 (lần cuối với `m = 1` giảm 1) ⇒ đúng `⌈m/2⌉` lần. Cách 2 (tham lam hai con trỏ) thực hiện chính các hoán đổi này trên xâu gốc; mục 4 và mục 12 kiểm chứng bằng máy, kể cả so với BFS.

**Công thức đếm** (cách 3, 4) tính đúng `m`: ngăn xếp loại mọi cặp khớp, còn lại `]^m[^m`; biến `open_` mô phỏng ngăn xếp mà không lưu các `]` thừa.

Kiểm chứng chặn dưới bằng máy: với mọi hoán đổi có thể trên nhiều xâu ngẫu nhiên, giá trị `min` tiền tố tăng **tối đa 2**.

In [15]:
rnd = random.Random(10)
max_gain = 0
for _ in range(300):
    h = rnd.randint(1, 6); t = list('[' * h + ']' * h); rnd.shuffle(t); t = ''.join(t)
    base = min([0] + prefix_balances(t))
    for i in range(len(t)):
        for j in range(i + 1, len(t)):
            gain = min([0] + prefix_balances(swap(t, i, j))) - base
            max_gain = max(max_gain, gain)
print("mức tăng lớn nhất của min tiền tố sau một hoán đổi:", max_gain)
assert max_gain == 2

mức tăng lớn nhất của min tiền tố sau một hoán đổi: 2


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. BFS | `O(C(n, n/2) · n³)` | hàm mũ |
| 2. Tham lam hai con trỏ | `O(n)` | `O(n)` |
| 3. Ngăn xếp | `O(n)` | `O(n)` |
| 4. Một biến đếm | `O(n)` | `O(1)` |

In [16]:
rnd = random.Random(1963)
half = 5 * 10**5
big = list('[' * half + ']' * half); rnd.shuffle(big); big = ''.join(big)
for name, fn in [("greedy", min_swaps_greedy), ("stack", min_swaps_stack), ("counter", min_swaps_counter), ("min_prefix", min_swaps_min_prefix), ("Solution", Solution().minSwaps)]:
    t0 = time.perf_counter(); r = fn(big); dt = time.perf_counter() - t0
    print(f"{name:<11} n=10^6: {r:>6}  {dt * 1000:7.1f} ms")

greedy      n=10^6:    129     46.0 ms
stack       n=10^6:    129     31.5 ms
counter     n=10^6:    129     29.9 ms
min_prefix  n=10^6:    129     42.0 ms
Solution    n=10^6:    129     29.7 ms


## 12. Bộ test

In [17]:
ALL = {
    "greedy": min_swaps_greedy,
    "stack": min_swaps_stack,
    "counter": min_swaps_counter,
    "min_prefix": min_swaps_min_prefix,
    "Solution": lambda s: Solution().minSwaps(s),
}

def check(s, exp=None, brute=True):
    ref = min_swaps_bfs(s) if brute else min_swaps_counter(s)
    for name, fn in ALL.items():
        assert fn(s) == ref, (name, s, fn(s), ref)
    if exp is not None:
        assert ref == exp, (s, ref, exp)
    return ref

for s_, exp in examples:
    check(s_, exp)
print("Ví dụ đề: OK")

for s_, exp in [("[[]]", 0), ("][", 1), ("]][[", 1), ("]]]][[[[", 2), ("]]]]][[[[[", 3), ("[]][][", 1), ("[[][]]", 0)]:
    check(s_, exp)
print("Edge case: OK")

cnt = 0
for n in range(2, 11, 2):
    for pos in combinations(range(n), n // 2):
        t = [']'] * n
        for p in pos:
            t[p] = '['
        check(''.join(t)); cnt += 1
print(f"Toàn bộ {cnt} xâu cân số ngoặc với n <= 10 so với BFS: OK")

Ví dụ đề: OK
Edge case: OK
Toàn bộ 350 xâu cân số ngoặc với n <= 10 so với BFS: OK


In [18]:
rnd = random.Random(2007)
for _ in range(300):
    h = rnd.randint(1, 2000)
    t = list('[' * h + ']' * h); rnd.shuffle(t); t = ''.join(t)
    check(t, brute=False)
    # tham lam phải dựng được xâu cân bằng với đúng số lần đó
    assert min_swaps_greedy(t) == min_swaps_counter(t)
print("300 xâu ngẫu nhiên cỡ lớn: các cách khớp nhau: OK")

300 xâu ngẫu nhiên cỡ lớn: các cách khớp nhau: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về `m` (số ngoặc thừa) thay vì `⌈m/2⌉`

In [19]:
def min_swaps_bug_m(s):
    open_ = 0
    for c in s:
        if c == '[': open_ += 1
        elif open_: open_ -= 1
    return open_                                    # SAI: một lần đổi sửa được 2 cặp

print("']]][[[': sai =", min_swaps_bug_m("]]][[["), "| đúng =", Solution().minSwaps("]]][[["))

']]][[[': sai = 3 | đúng = 2


**Sửa:** `(m + 1) // 2`.

### Lỗi 2 — Làm tròn xuống `m // 2`

In [20]:
def min_swaps_bug_floor(s):
    open_ = 0
    for c in s:
        if c == '[': open_ += 1
        elif open_: open_ -= 1
    return open_ // 2                               # SAI với m lẻ

for t in ["][", "]]][[["]:
    print(f"{t}: sai = {min_swaps_bug_floor(t)}, đúng = {Solution().minSwaps(t)}")

][: sai = 0, đúng = 1
]]][[[: sai = 1, đúng = 2


**Vì sao sai:** với `m = 1` (`"]["`) cần 1 lần, `1 // 2 = 0`. **Sửa:** làm tròn **lên**.

### Lỗi 3 — Đếm mọi `]` đứng trước `[` đầu tiên / đếm tất cả `]` khi `bal < 0` mà không "reset"

In [21]:
def min_swaps_bug_count_neg(s):
    bal = neg = 0
    for c in s:
        bal += 1 if c == '[' else -1
        if bal < 0:
            neg += 1                                  # SAI: đếm số vị trí có số dư âm
    return (neg + 1) // 2

for t in ["]]][[[", "][]["]:
    print(f"{t}: sai = {min_swaps_bug_count_neg(t)}, đúng = {Solution().minSwaps(t)}")

]]][[[: sai = 3, đúng = 2
][][: sai = 1, đúng = 1


**Vì sao sai:** một `]` thừa làm **nhiều** tiền tố liên tiếp bị âm; cái cần là **độ sâu âm lớn nhất** (`−min`), không phải số vị trí âm. **Sửa:** `m = −min(prefix)` hoặc đếm `[` chưa ghép.

### Lỗi 4 — Tham lam đổi với `[` **gần nhất** bên phải thay vì xa nhất

In [22]:
def min_swaps_bug_nearest(s):
    t = list(s); bal = swaps = 0
    for i in range(len(t)):
        bal += 1 if t[i] == '[' else -1
        if bal < 0:
            j = i + 1
            while t[j] != '[':
                j += 1                                # SAI: '[' gần nhất
            t[i], t[j] = t[j], t[i]; swaps += 1; bal = 1
    return swaps

rnd = random.Random(4)
found = None
while found is None:
    h = rnd.randint(2, 6); t = list('[' * h + ']' * h); rnd.shuffle(t); t = ''.join(t)
    if min_swaps_bug_nearest(t) != Solution().minSwaps(t):
        found = t
print(f"phản ví dụ {found!r}: đổi với '[' gần nhất = {min_swaps_bug_nearest(found)}, đúng = {Solution().minSwaps(found)}")

phản ví dụ ']][][[': đổi với '[' gần nhất = 3, đúng = 1


**Vì sao sai:** đổi với `[` gần nhất chỉ nâng số dư trên một đoạn ngắn; các `]` thừa phía sau vẫn cần thêm hoán đổi. `[` xa nhất bên phải nâng số dư trên đoạn **dài nhất** (mọi tiền tố tới gần cuối). **Sửa:** con trỏ `j` từ cuối xâu đi lùi.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài ngoặc ⇒ nghĩ ngay **số dư tiền tố** hoặc **ngăn xếp loại cặp khớp**.
- Sau khi loại cặp khớp, phần lỗi có dạng chuẩn `]^m[^m` (hoặc `)^a(^b`) ⇒ đếm và dùng công thức.
- "Hoán đổi" sửa được 2 lỗi mỗi lần ⇒ `⌈m/2⌉`; "thêm ngoặc" (LeetCode 921) sửa 1 lỗi ⇒ `a + b`; "đảo một ngoặc" (LeetCode 1541/2116) có công thức khác.
- Luôn kiểm công thức bằng BFS/vét cạn với `n ≤ 10`.
- Chú ý làm tròn **lên**.

## 15. Biến thể của bài

1. **LeetCode 921 — Minimum Add to Make Parentheses Valid:** đếm `)` thừa `a` và `(` thừa `b` ⇒ `a + b`.
2. **Đảo chiều một ngoặc mỗi thao tác** (đổi `[` thành `]` hoặc ngược lại): với `]^a[^b` (a + b chẵn) ⇒ `⌈a/2⌉ + ⌈b/2⌉`.
3. **LeetCode 1541 — Minimum Insertions to Balance a Parentheses String** (mỗi `(` khớp `))`).
4. **Số ngoặc mỗi loại không bằng nhau:** có thể không có lời giải bằng hoán đổi.
5. **Hoán đổi chỉ hai ký tự kề nhau** (đếm số hoán đổi kề): bài khác hẳn (đếm nghịch thế / tham lam theo vị trí).

In [23]:
def min_add_valid(s):
    """LeetCode 921 với '()'."""
    open_ = extra_close = 0
    for c in s:
        if c == '(':
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra_close += 1
    return open_ + extra_close

def min_flips_to_balance(s):
    """Biến thể 2: mỗi thao tác đảo một ngoặc; giả sử len chẵn."""
    open_ = extra_close = 0
    for c in s:
        if c == '[':
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra_close += 1
    return (extra_close + 1) // 2 + (open_ + 1) // 2

def min_flips_bfs(s):
    from collections import deque as dq
    dist = {s: 0}; q = dq([s])
    while q:
        cur = q.popleft()
        if is_balanced(cur): return dist[cur]
        for i in range(len(cur)):
            nxt = cur[:i] + ('[' if cur[i] == ']' else ']') + cur[i + 1:]
            if nxt not in dist:
                dist[nxt] = dist[cur] + 1; q.append(nxt)

print("921 '())' ->", min_add_valid("())"), "| '(((' ->", min_add_valid("((("))
for n in (2, 4, 6, 8):
    for bits in range(1 << n):
        t = ''.join('[' if bits >> i & 1 else ']' for i in range(n))
        assert min_flips_to_balance(t) == min_flips_bfs(t)
print("Biến thể đảo ngoặc khớp BFS với mọi xâu độ dài <= 8")

921 '())' -> 1 | '(((' -> 3
Biến thể đảo ngoặc khớp BFS với mọi xâu độ dài <= 8


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS | mọi xâu qua hoán đổi | hàm mũ | hàm mũ | kiểm thử `n ≤ 10` |
| 2. Tham lam hai con trỏ | đổi `]` thừa với `[` xa nhất | `O(n)` | `O(n)` | khi cần **dựng** lời giải |
| 3. Ngăn xếp | còn lại `]^m[^m` | `O(n)` | `O(n)` | trực quan |
| 4. Một biến đếm | `m` = `[` chưa ghép | `O(n)` | `O(1)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minSwaps("]][[][")` và `("][][][")`?

**Bài 2 (tính tay):** Với `s = "]]]][[[["`, trace cách 2 và ghi xâu sau mỗi lần hoán đổi.

**Bài 3 (điền TODO):** Hoàn thiện `swap_plan(s)` trả về **danh sách cặp chỉ số** hoán đổi (theo thứ tự) để `s` cân bằng với số lần tối thiểu.

**Bài 4 (tự viết lại):** Viết lại cách 4 dựa trên **số dư tiền tố nhỏ nhất** nhưng duyệt từ **phải sang trái** (xem `]` là +1, `[` là −1).

**Bài 5 (mini challenge):** Cho xâu `[`/`]` độ dài chẵn với số lượng **có thể khác nhau**, mỗi thao tác là **hoán đổi** hoặc **đảo một ngoặc** (đều tốn 1). Viết `min_ops_mixed(s)` và kiểm bằng BFS.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def swap_plan(s):
    # TODO: như min_swaps_greedy, ghi (i, j) mỗi lần đổi
    pass

# Bài 4 — TODO
def min_swaps_from_right(s):
    # TODO: duyệt đảo ngược, ']' là +1, '[' là -1, theo dõi min
    pass

# Bài 5 — TODO
def min_ops_mixed(s):
    # TODO: rút gọn về ]^a [^b; thử dùng hoán đổi cho min(a, b) cặp... kiểm bằng BFS
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Loại cặp khớp.
- Mức 2: `"]][[]["` → còn `"]][["`?
- Mức 3: `m = 2` ⇒ 1; `"][][]["` ⇒ `m = 1` ⇒ 1.

**Bài 2**
- Mức 1: `i = 0` âm ⇒ đổi với vị trí 7.
- Mức 2: Xâu thành `[]]][[[]`.
- Mức 3: Tiếp tục tới `i = 2`.

**Bài 3**
- Mức 1: Lưu `(i, j)` trong nhánh `bal < 0`.
- Mức 2: Áp dụng lần lượt lên xâu gốc phải ra xâu cân bằng.
- Mức 3: Độ dài kế hoạch bằng `Solution()`.

**Bài 4**
- Mức 1: Đối xứng: đọc ngược và đổi vai ngoặc.
- Mức 2: `m` giống hệt (số `[` thừa ở cuối = số `]` thừa ở đầu).
- Mức 3: `(m + 1) // 2`.

**Bài 5**
- Mức 1: Sau rút gọn còn `]^a[^b`, `a + b` chẵn.
- Mức 2: Một hoán đổi sửa 2 lỗi kiểu "một `]` + một `[`"; đảo sửa 1 lỗi.
- Mức 3: Dùng hoán đổi cho `min(a, b)` cặp `]…[` (tốn `⌈min/2⌉`), phần dư `|a − b|` (chẵn) cùng một loại: đảo một nửa ⇒ `|a − b| / 2`. Kiểm bằng BFS (công thức `⌈a/2⌉ + ⌈b/2⌉` của bài chỉ-đảo là **sai** ở đây, ví dụ `"]["`).

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r1 = (Solution().minSwaps("]][[]["), Solution().minSwaps("][][]["))
print("Bài 1:", r1, "| phần không khớp:", unmatched("]][[]["), unmatched("][][]["))
assert r1 == (min_swaps_bfs("]][[]["), min_swaps_bfs("][][]["))

# Bài 2
trace_greedy("]]]][[[[")

# Bài 3
def swap_plan_sol(s):
    t = list(s); j = len(t) - 1; bal = 0; plan = []
    for i in range(len(t)):
        bal += 1 if t[i] == '[' else -1
        if bal < 0:
            while t[j] != '[':
                j -= 1
            t[i], t[j] = t[j], t[i]; plan.append((i, j)); bal = 1
    return plan

rnd = random.Random(3)
for _ in range(500):
    h = rnd.randint(1, 40); t = list('[' * h + ']' * h); rnd.shuffle(t); t = ''.join(t)
    plan = swap_plan_sol(t)
    u = t
    for i, j in plan:
        u = swap(u, i, j)
    assert is_balanced(u) and len(plan) == Solution().minSwaps(t)
print("Bài 3 OK | ví dụ 2:", swap_plan_sol("]]][[["))

Bài 1: (1, 1) | phần không khớp: ]][[ ][
  i=0: bal= 1 -> âm! đổi 0<->7  t=[]]][[[] swaps=1
  i=1: bal= 0                   t=[]]][[[] swaps=1
  i=2: bal= 1 -> âm! đổi 2<->6  t=[][][[]] swaps=2
  i=3: bal= 0                   t=[][][[]] swaps=2
  i=4: bal= 1                   t=[][][[]] swaps=2
  i=5: bal= 2                   t=[][][[]] swaps=2
  i=6: bal= 1                   t=[][][[]] swaps=2
  i=7: bal= 0                   t=[][][[]] swaps=2
  kết quả: [][][[]] cân bằng: True
Bài 3 OK | ví dụ 2: [(0, 5), (2, 4)]


In [26]:
# Bài 4
def min_swaps_from_right_sol(s):
    bal = low = 0
    for c in reversed(s):
        bal += 1 if c == ']' else -1
        low = min(low, bal)
    return (-low + 1) // 2

# Bài 5
def min_ops_mixed_sol(s):
    open_ = extra_close = 0
    for c in s:
        if c == '[':
            open_ += 1
        elif open_:
            open_ -= 1
        else:
            extra_close += 1
    a, b = extra_close, open_                 # phần lỗi ]^a [^b, a + b chẵn
    m = min(a, b)
    return (m + 1) // 2 + (max(a, b) - m) // 2    # hoán đổi cho phần "cặp", đảo cho phần dư

def min_ops_mixed_bfs(s):
    dist = {s: 0}; q = deque([s])
    while q:
        cur = q.popleft()
        if is_balanced(cur):
            return dist[cur]
        n = len(cur)
        nxts = [cur[:i] + ('[' if cur[i] == ']' else ']') + cur[i + 1:] for i in range(n)]
        nxts += [swap(cur, i, j) for i in range(n) for j in range(i + 1, n) if cur[i] != cur[j]]
        for nx in nxts:
            if nx not in dist:
                dist[nx] = dist[cur] + 1; q.append(nx)

rnd = random.Random(5)
for _ in range(2000):
    h = rnd.randint(1, 300); t = list('[' * h + ']' * h); rnd.shuffle(t); t = ''.join(t)
    assert min_swaps_from_right_sol(t) == Solution().minSwaps(t)
for n in (2, 4, 6, 8):
    for bits in range(1 << n):
        t = ''.join('[' if bits >> i & 1 else ']' for i in range(n))
        assert min_ops_mixed_sol(t) == min_ops_mixed_bfs(t), t
print("Bài 4, Bài 5 OK | min_ops_mixed(']]]]') =", min_ops_mixed_sol("]]]]"), "| (']][[') =", min_ops_mixed_sol("]][["))

Bài 4, Bài 5 OK | min_ops_mixed(']]]]') = 2 | (']][[') = 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm ngoặc thừa":**

```python
open_ = extra_close = 0
for c in s:
    if c == OPEN:
        open_ += 1
    elif open_:
        open_ -= 1          # ghép cặp
    else:
        extra_close += 1    # ngoặc đóng thừa
# bài này: extra_close == open_ == m, đáp án (m + 1) // 2
```

**Checklist:**
- [ ] `m` là **số** ngoặc không khớp (không phải số vị trí có số dư âm)?
- [ ] Đáp án làm tròn **lên** `(m + 1) // 2`?
- [ ] Khi dựng lời giải: đổi với `[` **xa nhất** bên phải?
- [ ] `O(1)` bộ nhớ cho `n = 10⁶` (không cần list/ngăn xếp)?
- [ ] Đã kiểm với BFS cho mọi xâu `n ≤ 10`?
- [ ] Chữ ký đúng: `def minSwaps(self, s: str) -> int`?